# MNIST demo with CREST 

In [ ]:
# Save the start time
import time
start_time = time.time()

# Allow crest to be imported
%cd -q ..

#### Load MNIST Datasets

In [ ]:
from crest import DataServer

# Load datasets from server
ds_train,ds_valid,ds_test,ds_pred = DataServer.load('mnist')

# Get input/output labels
[INP,OUT] = ds_train.labels

# Num classes
n_classes = 10
patch_size = 28

#### Create the HierarchalTensorGraph

In [ ]:
from tensorflow.keras.layers import Conv2D, Dropout, Flatten, Dense, Lambda
import tensorflow as tf
from crest import HierarchalTensorGraph as HTG

def crest_layer(layer):
    """ Wrap a tensorflow layer in a HierarchalTensorGraph """
    graph = HTG(lambda d: {OUT: layer(d[INP])}, name=layer.name)
    #set inputs/outputs
    graph.inputs = {INP:None}
    graph.outputs = {OUT:None}
    return graph

# Define the tensorflow layers, and convert to HTG objects
layers = [crest_layer(layer) for layer in [
    Lambda(lambda x: tf.cast(x, tf.float32)),         # Cast dtype
    Lambda(lambda x: tf.expand_dims(x, axis=-1)),     # Add feature dimension
    Conv2D(10, 3, padding='same', activation='relu'), # 28x28x1 -> 28x28x10
    Flatten(),                                        # 28x28x10 -> 7840
    Dropout(0.3),                                     # 30% of data dropped
    Dense(n_classes, activation='softmax'),           # Class probabilities
]]

# Rename outputs to use as inputs (except the last layer)
for i in layers[:-1]:
    i.rename_io(outputs_map={OUT:INP})
    
# Create the CREST graph: ensure tensor shapes are specified for the inputs and outputs
graph = HTG(
    name='MNIST',
    inputs={INP : tf.TensorSpec(shape=[None,patch_size, patch_size])},
    outputs={OUT : tf.TensorSpec(shape=[None,10])}
)
print(list(map(graph.add_edge, ['input']+layers, layers+['output'])))

#### Build a model using the HTG

In [ ]:
from crest import Model

# Define a custom metric if desired
def custom_metric(y_true, y_pred):
    squared_difference = tf.square(y_true - y_pred)
    return tf.reduce_mean(squared_difference, axis=-1)

# Set build options
build_kwargs= {
    'optimizer' : 'Adam', 
    'loss'      : 'sparse_categorical_crossentropy', 
    'metrics'   : ['sparse_categorical_accuracy', custom_metric]
}

# Create anad build a model using the HTG
model = Model(graph)
model.build(**build_kwargs)

Traceback (most recent call last):
  File "/Users/ccruz/projects/terrahydro/development/crest/crest/utils/Metrics.py", line 35, in get_callbacks
    keras_avail = tf.keras.metrics.get(m)
  File "/Users/ccruz/miniconda3/envs/crest_cpu/lib/python3.10/site-packages/keras/metrics/__init__.py", line 181, in get
    return deserialize(str(identifier))
  File "/Users/ccruz/miniconda3/envs/crest_cpu/lib/python3.10/site-packages/keras/metrics/__init__.py", line 136, in deserialize
    return deserialize_keras_object(
  File "/Users/ccruz/miniconda3/envs/crest_cpu/lib/python3.10/site-packages/keras/utils/generic_utils.py", line 769, in deserialize_keras_object
    raise ValueError(
ValueError: Unknown metric function: sparce_categorical_accuracy. Please ensure this object is passed to the `custom_objects` argument. See https://www.tensorflow.org/guide/keras/save_and_serialize#registering_the_custom_object for details.

[]
Epoch 1/5
  1/250 [..............................] - ETA: 36s - loss: 71.7

2023-11-20 10:08:08.867283: W tensorflow/core/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


250/250 [==============================] - 2s 9ms/step - loss: 3.2148 - val_loss: 0.4592
Epoch 2/5
250/250 [==============================] - 2s 9ms/step - loss: 0.4799 - val_loss: 0.2707
Epoch 3/5
250/250 [==============================] - 2s 9ms/step - loss: 0.2505 - val_loss: 0.2172
Epoch 4/5
250/250 [==============================] - 2s 9ms/step - loss: 0.1575 - val_loss: 0.1869
Epoch 5/5
250/250 [==============================] - 2s 9ms/step - loss: 0.1148 - val_loss: 0.1599


#### Train the model

In [ ]:
# Create the training and validation batchers
from crest.data.Batcher import Batcher

batch_kwargs = {
    'batch_size' : 200,            # 200 samples in each batch
    'features'   : [[INP], [OUT]], # [{'image': ...}, {'class': ...}]
    'repeat'     : True,           # Yield indefinitely
    'workers'    : 0               # Number of background processes
}

batch_train = Batcher(ds_train, **batch_kwargs)
batch_kwargs['shuffle'] = False    # Turn shuffle off for prediction
batch_valid = Batcher(ds_valid,  **batch_kwargs)

with batch_train as train, batch_valid as valid:
    model.fit(train, **{
    'validation_data'  : valid,
    'steps_per_epoch'  : 250, # 50k samples / 200 samples per batch
    'validation_steps' : 50,  # 10k samples / 200 samples per batch
    'epochs'           : 5,
    'verbose'          : 1
    })

#### Evaluate the model

In [9]:
# Evaluate the model on the test set
batch_test = Batcher(ds_test, **batch_kwargs)
evaluate_kwargs= {
    'verbose' : 1,
    'return_dict' : True
}

with batch_test as batcher:
    res = model.evaluate(batcher, verbose=1, return_dict=1)

1/1 [==============================] - 0s 13ms/step - loss: 0.0481
{'loss': 0.048074282705783844}


Accuracy on the full test set should be > ~97%.

### Make predictions

In [11]:
# Create batcher for predictions
batch_pred = Batcher(ds_pred, **{  # Reset batch iterator
    'batch_size' : 200,
    'features'   : [INP],
    'repeat'     : True,
    'workers'    : 0,
    'shuffle'    : False
}) 

# Set kwargs for Model.predict() (can be the same as Keras.predict)
pred_kwargs = {
    'batch_size' : 200,
    'verbose'    : False,
    'steps'      : 10000//200
}

# Generate predictions
with batch_pred as batcher:
    model_pred_batch = model.predict(batch_pred, **pred_kwargs) # Using a batcher
    model_pred_dataset = model.predict(ds_pred,**pred_kwargs)   # Using a dataset
    print('Difference:', (model_pred_batch['class'] - model_pred_dataset['class']).sum())

#### Adding additional output (e.g., lat and lon) when making a predictions

In [14]:
# Create inputs with additional features lat and lon
import numpy as np
from crest.data.loading import StructuredDataset
x = ds_pred.data[0]
lat = np.arange(len(x),dtype=float)
lon = np.arange(len(x),dtype=float)
x_aux = (lat,lon,x)
ds_aux = StructuredDataset(*x_aux, labels=['lat','lon',INP])

# Specify additional output as coords when making predictions
model.predict(ds_aux,coords=['lat','lon'],**pred_kwargs)

In [ ]:
print('Elapsed time = ', time.time() - start_time, '[s]')